# Week 4 Capstone — Predicting Internship Selection Outcomes
**Intern:** Maryam Shaikh | **Program:** NextGen Learners — Data Science Internship

This notebook is written as a **report**, not just a script. It takes the cleaned applicant dataset from Week 1, adds a few simulated features, trains a simple machine learning model to predict whether an applicant is **Selected**, evaluates it honestly, and explains the result in plain language.

**Contents**
1. Setup
2. Load the cleaned Week 1 data
3. Extend the dataset with simulated features
4. Feature preparation
5. Train/test split
6. Train the model
7. Evaluate the model honestly
8. Interpret which features mattered
9. Executive Summary

## Step 1 — Setup
Libraries used across the whole notebook. Pandas and NumPy handle the data, Matplotlib draws charts, and scikit-learn provides the model and the evaluation metrics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay

## Step 2 — Load the cleaned Week 1 data
I start from the file my Week 1 notebook exported (`applicants_cleaned_by_maryamshaikh.csv`). Phone is read as text so leading zeros are not lost.

**Running in Google Colab?** Upload the CSV with the file button in the left sidebar first. **Running locally?** Keep the CSV in the same folder as this notebook.

In [ ]:
df = pd.read_csv('applicants_cleaned_by_maryamshaikh.csv', dtype={'Phone': str})
print("Shape:", df.shape)
df.head()

In [ ]:
print(df['Status'].value_counts())
print()
print(df['Domain Applied'].value_counts())

**What this shows:** the dataset is small, and `Status` has three values: **Selected**, **Rejected** and **Under Review**. "Under Review" means no final decision exists yet, so those applicants cannot teach the model anything (this matters in Step 4.2).

## Step 3 — Extend the dataset with simulated features

> ### ⚠️ Honesty note — please read
> The Week 1 dataset only contains basic applicant info (name, email, domain, university, status). To build a meaningful prediction model I need more input signals ("features"). I do **not** have real historical data for these, so **the 4 features below are SIMULATED — generated by code, not collected from real applicants.**
>
> | Feature | Meaning |
> |---|---|
> | `num_skills_listed` | How many relevant skills the applicant listed (1–8) |
> | `has_portfolio` | Whether they gave a GitHub/portfolio link (Yes/No) |
> | `prior_hackathon_participation` | Whether they joined a hackathon before (Yes/No) |
> | `statement_quality_score` | How strong their short statement was (1–10) |
>
> In a real NextGen Learners deployment, this data would have to be **actually collected** (e.g. a required skills field and a portfolio URL field on the application form) before the model could be trusted for any real decision.

**How I simulated them:** `np.random.seed(42)` makes the "random" values identical every time the notebook is re-run (reproducibility). The switch `LINK_TO_OUTCOME` controls how realistic the simulation is:
- `True` (default): `has_portfolio`, `prior_hackathon_participation` and `statement_quality_score` are made *somewhat more likely to be high* for Selected applicants, plus plenty of random noise — mimicking a world where strong applicants tend to have these signals. `num_skills_listed` stays purely random.
- `False`: every feature is purely random, unrelated to the outcome. The model should then perform close to chance, which is itself an honest result.

**Important consequence:** with `True`, the link between features and selection was *built in by me*. So if the model "discovers" that the statement score matters, that is **circular** — it found the pattern I planted. This shows the *method*, not a real finding about NextGen Learners.

In [ ]:
np.random.seed(42)          # makes the simulated data reproducible
LINK_TO_OUTCOME = True      # False = purely random features (no link to Status)

n = len(df)

if LINK_TO_OUTCOME:
    # strength: 1 = Selected, 0 = Rejected, 0.5 = Under Review (unknown outcome -> neutral)
    strength = np.where(df['Status'] == 'Selected', 1.0,
                np.where(df['Status'] == 'Rejected', 0.0, 0.5))
else:
    strength = np.full(n, 0.5)

df['num_skills_listed'] = np.random.randint(1, 9, size=n)   # purely random, 1-8

has_portfolio_yes = np.random.rand(n) < (0.25 + 0.35 * strength * LINK_TO_OUTCOME)
df['has_portfolio'] = np.where(has_portfolio_yes, 'Yes', 'No')

hackathon_yes = np.random.rand(n) < (0.20 + 0.15 * strength * LINK_TO_OUTCOME)
df['prior_hackathon_participation'] = np.where(hackathon_yes, 'Yes', 'No')

score = np.random.normal(loc=4.5 + 2.5 * strength * LINK_TO_OUTCOME, scale=1.8, size=n)
df['statement_quality_score'] = np.clip(np.round(score), 1, 10).astype(int)

df[['Applicant Name', 'Status', 'num_skills_listed', 'has_portfolio',
    'prior_hackathon_participation', 'statement_quality_score']].head(10)

In [ ]:
# Save the extended dataset so the simulated columns are documented and reproducible
df.to_csv('applicants_with_simulated_features.csv', index=False)
print("Saved: applicants_with_simulated_features.csv")

## Step 4 — Feature preparation: getting the data ready for a model
Machine learning models only understand **numbers**. They cannot interpret text like "Yes" or "Web Development", so this step turns everything into numeric form.

### 4.1 Convert Yes/No columns to 1/0
A model can work with 1/0 mathematically (e.g. how much does having a portfolio change the odds of selection?). It has no way to interpret the text "Yes".

In [ ]:
df['has_portfolio'] = df['has_portfolio'].map({'Yes': 1, 'No': 0})
df['prior_hackathon_participation'] = df['prior_hackathon_participation'].map({'Yes': 1, 'No': 0})
df[['has_portfolio', 'prior_hackathon_participation']].head()

### 4.2 Create the target column (what the model will predict)
The `target` column is **1 if Status is Selected, otherwise 0** (Rejected). 

**Under Review** applicants have no confirmed outcome, so there is nothing for the model to learn from. I drop them from the modelling data (`df_model`) rather than wrongly labelling them as "not selected".

In [ ]:
df['target'] = df['Status'].apply(lambda x: 1 if x == 'Selected' else 0)

df_model = df[df['Status'] != 'Under Review'].copy()

print("Rows before dropping Under Review:", len(df))
print("Rows used for modelling:          ", len(df_model))
print()
print(df_model['target'].value_counts().rename({1: 'Selected (1)', 0: 'Rejected (0)'}))

In [ ]:
counts = df_model['target'].value_counts().sort_index()
plt.figure(figsize=(5, 3.5))
plt.bar(['Rejected (0)', 'Selected (1)'], counts.values, color=['#9aa5b1', '#2a9d8f'])
plt.title('Class balance in the modelling data')
plt.ylabel('Number of applicants')
for i, v in enumerate(counts.values):
    plt.text(i, v + 0.3, str(v), ha='center')
plt.tight_layout()
plt.show()

**Why look at class balance?** If one outcome is much more common than the other, a model can look accurate just by always guessing the common one. That is why accuracy alone is not enough (see Step 7).

### 4.3 One-hot encode `Domain Applied`
Domain has several categories (Web Development, Data Science, …) with **no natural order**. Coding them 1, 2, 3 would wrongly tell the model that one domain is "more" than another. **One-hot encoding** instead creates a separate 0/1 column per domain.

`drop_first=True` removes one domain's column — that domain becomes the "default" (all zeros in the other domain columns). This avoids redundant information that can confuse some models and is standard practice.

In [ ]:
df_model = pd.get_dummies(df_model, columns=['Domain Applied'], drop_first=True, dtype=int)
print([c for c in df_model.columns if c.startswith('Domain Applied_')])

### 4.4 Select the final feature set
**Features used:** the 4 simulated features + the one-hot domain columns.

**Deliberately excluded:**
- `Applicant Name`, `Email`, `Phone` — these are **identifiers**, not meaningful predictive signals. Using them would be useless (and inappropriate).
- `University` — real data, but I excluded it to keep this first model simple, and because using it could encourage the model to favour certain universities, which is a bias risk (discussed in Step 7.3).
- `Application Date` — it has no clear link to applicant quality, and I want to avoid the model picking up accidental timing patterns.
- `Status` and `target` — the target is the answer; including it as a feature would be cheating (data leakage).

In [ ]:
feature_columns = ['num_skills_listed', 'has_portfolio',
                   'prior_hackathon_participation', 'statement_quality_score'] + \
                  [col for col in df_model.columns if col.startswith('Domain Applied_')]

X = df_model[feature_columns]
y = df_model['target']

print("Number of features:", X.shape[1])
print(feature_columns)

## Step 5 — Train/test split
**Why split at all?** If I train *and* evaluate on the same data, the model can simply memorise the answers instead of learning patterns — like grading a student on the exact questions they studied. A **test set** (data the model never sees during training) lets me check whether it works on new, unseen applicants.

**Why 80/20?** A common standard: 80% to train, 20% held back for honest evaluation. `random_state=42` makes the split reproducible. I also use `stratify=y` so that Selected/Rejected appear in the same proportion in both parts — important with a small dataset, otherwise the test set could by chance contain almost no Selected applicants.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:    ", len(X_test))
print()
print("Selected in training set:", int(y_train.sum()), "| Selected in test set:", int(y_test.sum()))

**Honest note on size:** the test set is very small (see the numbers above). One extra right or wrong prediction moves the percentages a lot, so the results in Step 7 should be read as **illustrations of the method**, not precise measurements.

## Step 6 — Train the model
I chose **Logistic Regression**: it is the standard, simple model for yes/no predictions (Selected vs Not Selected), and its results can be explained in plain language through its coefficients.

**What `.fit()` does:** this is the actual "learning" step. The model looks at the training features (`X_train`) alongside the correct answers (`y_train` — was each applicant really selected?) and adjusts itself internally to find patterns that connect the two.

*(A Decision Tree is the other acceptable choice. It is more visual, but needs `max_depth` to be limited to avoid **overfitting** — when a model memorises its small training set instead of learning general patterns. Logistic Regression is less prone to this on tiny datasets.)*

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Model trained. Predictions made for", len(y_pred), "test applicants.")

## Step 7 — Evaluate the model honestly

### 7.1 Accuracy
**Accuracy** = the percentage of test applicants whose outcome (Selected / Not Selected) the model predicted correctly.

**Limitation:** if one outcome is much more common, accuracy can mislead — a model that *always* predicts the common outcome can still score high while being useless. So I also print a **baseline**: the accuracy of always guessing the majority outcome. A useful model must beat it.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
baseline = max(y_test.mean(), 1 - y_test.mean())

print(f"Accuracy: {accuracy:.2%}")
print(f"Baseline (always guess the most common outcome): {baseline:.2%}")

### 7.2 Precision, Recall and the Confusion Matrix
- **Precision:** *"Of all applicants my model predicted would be Selected, what percentage were actually Selected?"* High precision = fewer false promises.
- **Recall:** *"Of all applicants who were actually Selected, what percentage did my model correctly catch?"* High recall = fewer missed strong candidates.
- **Confusion matrix:** a 2×2 grid showing how many predictions fell into each of 4 buckets: correctly predicted Selected, correctly predicted Not Selected, wrongly predicted Selected (false positive), and wrongly predicted Not Selected (false negative).

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Selected', 'Selected'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Predicted vs Actual Selection Outcome')
plt.show()

In [ ]:
# The same results in plain English, using my actual numbers
tn, fp, fn, tp = cm.ravel()

print(f"Test set: {len(y_test)} applicants, of whom {tp + fn} were actually Selected.")
print(f"- The model correctly identified {tp} of the {tp + fn} actually-selected applicants (recall {recall:.0%}).")
print(f"- Of the {tp + fp} applicants it predicted would be Selected, {tp} really were (precision {precision:.0%}).")
print(f"- It wrongly flagged {fp} not-selected applicant(s) as likely to be selected (false positives).")
print(f"- It missed {fn} applicant(s) who were actually selected (false negatives).")
print(f"- It correctly identified {tn} of the {tn + fp} not-selected applicants.")

**How to read this:** compare the accuracy with the baseline above. If the model barely beats (or does not beat) the baseline, it has learned little. Check that precision and recall are both reasonable — one can look good while the other is poor.

### 7.3 Model limitations (honest assessment)
- **Small, partly simulated data.** The dataset has only a few dozen labelled applicants, and **4 of the features were artificially generated**, not real historical behaviour. Any pattern the model "finds" in them was either planted by my simulation or is noise.
- **Tiny test set = unstable results.** With so few test rows, re-splitting the data differently can change the numbers noticeably (the check below shows by how much). These figures illustrate the *method*; they are not a production-ready guarantee.
- **Possible bias.** A model learns from past decisions. If past selections favoured certain universities, domains or backgrounds for reasons unrelated to merit, a model trained on them would learn and **repeat** that pattern. This is a genuine, important concern with any predictive hiring or selection tool and must never be glossed over.
- **Decision support, not decision maker.** This model should only ever *assist* human reviewers. It must not replace human review of applications.

### 7.4 How stable are these numbers? (extra check)
To see how much the result depends on the particular split, I repeat the train/test split 20 times with different random seeds and look at the spread of accuracy.

In [ ]:
accs = []
for seed in range(1, 21):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    m = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    accs.append(accuracy_score(yte, m.predict(Xte)))

print(f"Accuracy over 20 different splits: mean {np.mean(accs):.2%}, "
      f"lowest {np.min(accs):.2%}, highest {np.max(accs):.2%}")

**Takeaway:** the gap between the lowest and highest accuracy shows how much a single small test set can mislead. This is why I describe the results cautiously and avoid strong claims.

## Step 8 — Interpret: which features mattered most?
For Logistic Regression I look at the **coefficients**. A **positive** coefficient means that as the feature goes up, the model predicts "Selected" as more likely; a **negative** one means the opposite. The `Odds Ratio` column (e^coefficient) says the same thing as a multiplier: above 1 raises the odds of selection, below 1 lowers them.

*Caution:* the features are on different scales (a 1–10 score versus a 0/1 flag), so coefficient sizes are only a rough guide to importance.

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

coefficients['Odds Ratio'] = np.exp(coefficients['Coefficient']).round(2)
print(coefficients.to_string(index=False))

In [ ]:
plot_df = coefficients.sort_values(by='Coefficient')
colors = ['#2a9d8f' if c > 0 else '#e76f51' for c in plot_df['Coefficient']]

plt.figure(figsize=(8, 4.5))
plt.barh(plot_df['Feature'], plot_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('What the model learned: feature coefficients')
plt.xlabel('Coefficient (positive = more likely Selected)')
plt.tight_layout()
plt.show()

In [ ]:
top = coefficients.iloc[0]
bottom = coefficients.iloc[-1]
print(f"Strongest positive feature: {top['Feature']} (coefficient {top['Coefficient']:.2f})")
print(f"Strongest negative feature: {bottom['Feature']} (coefficient {bottom['Coefficient']:.2f})")

**Interpretation (edit this paragraph after you see your own numbers):**
In this dataset, the features with the largest positive coefficients — typically `statement_quality_score` and `has_portfolio` — were the strongest predictors of selection, while `num_skills_listed` had little influence. This suggests that, *in this dataset*, the quality of the written statement and evidence of project work mattered more than the raw count of listed skills.

**Reminder:** several features were simulated, and in the default setting the link to selection was built into the simulation. This is **what the model learned from this dataset**, not a real finding about how NextGen Learners actually selects interns. It would need to be validated on real historical data.

## Step 9 — Executive Summary
*Run the small cell below to get the numbers, then fill them into the summary. Keep the summary in plain language — no code or technical jargon.*

In [ ]:
core = ['num_skills_listed', 'has_portfolio', 'prior_hackathon_participation', 'statement_quality_score']
core_top = coefficients[coefficients['Feature'].isin(core)].head(2)['Feature']

print(f"About {accuracy:.0%} of applicants in the test group were predicted correctly.")
print(f"Top applicant-level factors linked to selection: {', '.join(core_top)}")
print(f"Test group size: {len(y_test)} applicants (training group: {len(y_train)})")

## Executive Summary

**What we did:** We built a simple prediction model using applicant data to explore which factors are associated with being selected for the NextGen Learners internship program.

**What we found:** The model correctly predicted the selection outcome for about **[X]%** of applicants in our test group. The strongest factors associated with selection were **[e.g. the strength of the applicant's written statement and whether they provided a portfolio/GitHub link]**.

**Important caveats:** This analysis used a small, partly simulated dataset — several factors (like skill count and statement quality) were artificially generated for this exercise rather than pulled from real historical records. Before using anything like this for real decisions, NextGen Learners would need to collect this information consistently from real applications and check the model on a much larger, real dataset. The model should be treated as a tool to assist reviewers, never as an automatic replacement for human judgment — especially because it could unintentionally repeat biases from past decisions.

**Recommendation:** **[One clear, specific action — e.g. "Add a required portfolio/GitHub link field and a short skills field to the application form, so this information is collected consistently and a future model can be built on real data."]**